In [7]:
import os
import cv2 as cv
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy

In [8]:
from pathlib import Path

# Folder input: hasil preprocessing
PREPROCESSING_DIR = Path("../preprocessing_output")

# Folder output: hasil ekstraksi fitur CSV
OUTPUT_DIR = Path("../hasil_ekstraksi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Folder hasil preprocessing yang akan diekstraksi fiturnya
PREPO_CONFIG = {
    "prepo1_resize+grayscale": "hasil_ekstraksi_prepo1.csv",
    "prepo2_resize+grayscale+median": "hasil_ekstraksi_prepo2.csv",
    "prepo3_resize+grayscale+median+equ": "hasil_ekstraksi_prepo3.csv",
    "prepo4_resize+grayscale+median+sobel": "hasil_ekstraksi_prepo4.csv",
    "prepo5_resize+grayscale+median+sobel+thresholding": "hasil_ekstraksi_prepo5.csv"
}

VALID_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

print("Folder preprocessing:", PREPROCESSING_DIR.resolve())
print("Folder hasil ekstraksi:", OUTPUT_DIR.resolve())

for prepo_name in PREPO_CONFIG:
    prepo_path = PREPROCESSING_DIR / prepo_name
    print(prepo_name, "->", "ADA" if prepo_path.exists() else "TIDAK ADA")

Folder preprocessing: D:\AAAAA KULIAH\SEM 4\PECEDE\Project-PCD-Kelompok-16\preprocessing_output
Folder hasil ekstraksi: D:\AAAAA KULIAH\SEM 4\PECEDE\Project-PCD-Kelompok-16\hasil_ekstraksi
prepo1_resize+grayscale -> ADA
prepo2_resize+grayscale+median -> ADA
prepo3_resize+grayscale+median+equ -> ADA
prepo4_resize+grayscale+median+sobel -> ADA
prepo5_resize+grayscale+median+sobel+thresholding -> ADA


In [9]:
def glcm(image, derajat):
    if derajat == 0:
        angles = [0]
    elif derajat == 45:
        angles = [np.pi / 4]
    elif derajat == 90:
        angles = [np.pi / 2]
    elif derajat == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Derajat harus salah satu dari: 0, 45, 90, 135.")

    matriks_glcm = graycomatrix(
        image,
        distances=[1],
        angles=angles,
        levels=256,
        symmetric=True,
        normed=True
    )

    return matriks_glcm


def safe_value(value):
    value = float(value)
    if np.isnan(value) or np.isinf(value):
        return 0.0
    return value


def correlation(matriks):
    return safe_value(graycoprops(matriks, 'correlation')[0, 0])


def dissimilarity(matriks):
    return safe_value(graycoprops(matriks, 'dissimilarity')[0, 0])


def homogenity(matriks):
    return safe_value(graycoprops(matriks, 'homogeneity')[0, 0])


def contrast(matriks):
    return safe_value(graycoprops(matriks, 'contrast')[0, 0])


def ASM(matriks):
    return safe_value(graycoprops(matriks, 'ASM')[0, 0])


def energy(matriks):
    return safe_value(graycoprops(matriks, 'energy')[0, 0])


def entropyGlcm(matriks):
    return safe_value(entropy(matriks.ravel()))

In [10]:
def extract_glcm_features(image):
    angles = [0, 45, 90, 135]
    features = {}

    for angle in angles:
        matriks = glcm(image, angle)

        features[f"Contrast{angle}"] = contrast(matriks)
        features[f"Homogeneity{angle}"] = homogenity(matriks)
        features[f"Dissimilarity{angle}"] = dissimilarity(matriks)
        features[f"Entropy{angle}"] = entropyGlcm(matriks)
        features[f"ASM{angle}"] = ASM(matriks)
        features[f"Energy{angle}"] = energy(matriks)
        features[f"Correlation{angle}"] = correlation(matriks)

    return features

In [11]:
def extract_features_from_preprocessing_folder(prepo_name, output_csv_name):
    prepo_path = PREPROCESSING_DIR / prepo_name

    if not prepo_path.exists():
        print(f"Folder tidak ditemukan: {prepo_path}")
        return None

    rows = []

    for class_folder in sorted(prepo_path.iterdir()):
        if not class_folder.is_dir():
            continue

        label = class_folder.name

        image_files = [
            file for file in sorted(class_folder.iterdir())
            if file.suffix.lower() in VALID_EXTENSIONS
        ]

        print(f"{prepo_name} | {label}: {len(image_files)} gambar")

        for image_path in image_files:
            image = cv.imread(str(image_path), cv.IMREAD_GRAYSCALE)

            if image is None:
                print(f"Gagal membaca gambar: {image_path}")
                continue

            image = image.astype(np.uint8)

            feature_data = extract_glcm_features(image)

            row = {
                "Filename": image_path.name,
                "Label": label,
                "Preprocessing": prepo_name
            }

            row.update(feature_data)
            rows.append(row)

    df = pd.DataFrame(rows)

    output_csv_path = OUTPUT_DIR / output_csv_name
    df.to_csv(output_csv_path, index=False)

    print(f"CSV berhasil disimpan: {output_csv_path}")
    print(f"Jumlah data: {len(df)}")

    return df

In [12]:
hasil_ekstraksi = {}

for prepo_name, output_csv_name in PREPO_CONFIG.items():
    print("\n" + "=" * 80)
    print(f"Ekstraksi fitur: {prepo_name}")
    print("=" * 80)

    df = extract_features_from_preprocessing_folder(prepo_name, output_csv_name)

    if df is not None:
        hasil_ekstraksi[prepo_name] = df

print("\nSemua ekstraksi fitur selesai.")


Ekstraksi fitur: prepo1_resize+grayscale
prepo1_resize+grayscale | catterpillar: 70 gambar
prepo1_resize+grayscale | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo1.csv
Jumlah data: 140

Ekstraksi fitur: prepo2_resize+grayscale+median
prepo2_resize+grayscale+median | catterpillar: 70 gambar
prepo2_resize+grayscale+median | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo2.csv
Jumlah data: 140

Ekstraksi fitur: prepo3_resize+grayscale+median+equ
prepo3_resize+grayscale+median+equ | catterpillar: 70 gambar
prepo3_resize+grayscale+median+equ | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo3.csv
Jumlah data: 140

Ekstraksi fitur: prepo4_resize+grayscale+median+sobel
prepo4_resize+grayscale+median+sobel | catterpillar: 70 gambar
prepo4_resize+grayscale+median+sobel | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo4.csv
Jumlah data: 140

Ekstraksi fitur: prepo